# AFFECTA AI — RAF-DB Training (EXP-007 EffNet-B3 → EXP-008 ViT-Small)

Accuracy-first fine-tuning on RAF-DB official split with ImageNet-pretrained backbones.

- **GPU**: use a T4 x2 (or P100) accelerator from Run settings.
- **Data**: attach the public dataset `shuvoalok/raf-db-dataset` (Settings → Input) OR run the download cell below.
- **Outputs**: checkpoints + test_metrics land in `/kaggle/working/experiments/...` and get zipped to `/kaggle/working/affecta_results.zip`.

In [ ]:
import os, sys, subprocess, json, time
print('python', sys.version)
import torch
HAS_GPU = torch.cuda.is_available()
print('cuda available:', HAS_GPU, '| devices:', torch.cuda.device_count())
if HAS_GPU:
    print('device:', torch.cuda.get_device_name(0))
else:
    nvs = (os.popen('nvidia-smi -L 2>/dev/null').read() or 'N/A').strip()
    print('nvidia-smi:', nvs, '| torch build:', torch.__version__)
    print('NOTE: this is a CPU run -> smoke-testing mode (no real training).')
    print(json.dumps({'has_gpu': HAS_GPU, 'cuda': bool(HAS_GPU)}))


## 1. Install dependencies

Kaggle already ships torch + torchvision. We add timm (backbone zoo), plus pandas/sklearn which the trainer uses.

In [ ]:
!pip install -q timm pandas scikit-learn Pillow
import timm
print('timm', timm.__version__)

## 2. Get the training code

The repo is attached as the private dataset `zopevipul/affecta-ai-code` (Settings → Input). It mounts under `/kaggle/input/datasets/` as read-only, so we copy it into `/kaggle/working` to get a writable copy (needed for `experiments/` outputs).

In [ ]:
import shutil
from pathlib import Path

SRC = None
for root, dirs, files in os.walk('/kaggle/input'):
    if (Path(root) / 'ml' / 'training' / 'train_expression.py').is_file() \
            and (Path(root) / 'scripts' / 'prepare_rafdb.py').is_file():
        SRC = root
        break
if SRC is None:
    print('DEBUG: /kaggle/input tree:')
    for r, ds, fs in os.walk('/kaggle/input'):
        print('  ', r, fs[:5])
assert SRC is not None, 'repo code input not found under /kaggle/input'
print('repo input at:', SRC)

DST = '/kaggle/working/affecta-ai'
if not os.path.exists(DST):
    shutil.copytree(SRC, DST)
assert os.path.isdir(DST + '/ml'), f'missing ml/ in {SRC}: {sorted(os.listdir(SRC))[:15]}'
os.chdir(DST)
sys.path.insert(0, DST)
print('cwd:', os.getcwd())
print('repo files:', sorted(os.listdir('.'))[:14])

## 3. Data — RAF-DB official split

Two options (pick one):
1. **Attach as input** (recommended): Settings → Input → Datasets → add `shuvoalok/raf-db-dataset`. It appears at `/kaggle/input/raf-db-dataset`.
2. **Download in-cell** via the Kaggle API (works with the same Kaggle credentials used here) or a direct public zip.
Then run `scripts/prepare_rafdb.py --source kaggle` which reproduces the exact train/val/test layout the trainer expects.

In [ ]:
from pathlib import Path

raw = None
for root, dirs, files in os.walk('/kaggle/input'):
    p = Path(root)
    if (p / 'DATASET' / 'train').is_dir():
        raw = p
        break
print('data root:', raw)
if raw is None:
    print('No mounted RAF-DB input found; listing /kaggle/input:')
    for r, ds, fs in os.walk('/kaggle/input'):
        print('  ', r)
    raise SystemExit(1)
print('train dirs:', sorted((raw/'DATASET'/'train').iterdir())[:3])

In [ ]:
!cd /kaggle/working/affecta-ai && PYTHONPATH=/kaggle/working/affecta-ai python scripts/prepare_rafdb.py --raw {raw} --out /kaggle/working/processed/rafdb --source kaggle --val-fraction 0.15
print('---');
!cat /kaggle/working/processed/rafdb/summary.json

In [ ]:
import torch
from ml.models.backbones import build_expression_model, input_size_for

results = {}
for name, exp in [('efficientnet_b3', 'EXP-007'), ('vit_small_patch16_224', 'EXP-008')]:
    model = build_expression_model(backbone=name, pretrained=True, dropout=0.5).eval()
    inp = torch.randn(2, 3, input_size_for(name), input_size_for(name))
    n_params = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    with torch.no_grad():
        y = model(inp)
    results[exp] = {'backbone': name, 'params': n_params / 1e6, 'trainable': n_train / 1e6,
                    'out_shape': list(y.shape), 'input_size': input_size_for(name)}
    print(f"{exp}: {name} ok, out={tuple(y.shape)} trainable={n_train/1e6:.1f}M")
print('SMOKE_OK', json.dumps(results))

## 4. EXP-007 — EfficientNet-B3 (input 300, strong aug)

Command mirrors the local run pattern: staged fine-tune, ImageNet-pretrained, class-weighted CE + label smoothing 0.05, warm-start from timm pretrained weights (default `pretrained=True`).

In [ ]:
if not HAS_GPU:
    print('CPU run - skipping EXP-007 real training (smoke results above).')
else:
    !PYTHONPATH=/kaggle/working/affecta-ai python -m ml.training.train_expression \
        --backbone efficientnet_b3 \
        --experiment-id EXP-007-EFFICIENTNET-B3 \
        --data-dir /kaggle/working/processed/rafdb \
        --epochs 30 --batch-size 32 --lr 1e-4 --weight-decay 1e-4 \
        --augmentation strong --amp
    print('EXP-007 done')

In [ ]:
import json, glob
exp7 = glob.glob('/kaggle/working/affecta-ai/experiments/EXP-007*')
if exp7:
    print('EXP-007 dir:', exp7[0])
    m = json.load(open(f'{exp7[0]}/test_metrics.json'))
    print('test_acc=%.2f%% test_macro_f1=%.2f%%' % (m['test_acc'], m['test_macro_f1']))
else:
    print('EXP-007 not run (CPU smoke mode)')


## 5. EXP-008 — ViT-Small (input 224, strong aug)

In [ ]:
if not HAS_GPU:
    print('CPU run - skipping EXP-008 real training (smoke results above).')
else:
    !PYTHONPATH=/kaggle/working/affecta-ai python -m ml.training.train_expression \
        --backbone vit_small_patch16_224 \
        --experiment-id EXP-008-VIT-SMALL \
        --data-dir /kaggle/working/processed/rafdb \
        --epochs 30 --batch-size 32 --lr 1e-4 --weight-decay 1e-4 \
        --augmentation strong --amp
    print('EXP-008 done')

In [ ]:
import json, glob
exp8 = glob.glob('/kaggle/working/affecta-ai/experiments/EXP-008*')
if exp8:
    print('EXP-008 dir:', exp8[0])
    m = json.load(open(f'{exp8[0]}/test_metrics.json'))
    print('test_acc=%.2f%% test_macro_f1=%.2f%%' % (m['test_acc'], m['test_macro_f1']))
else:
    print('EXP-008 not run (CPU smoke mode)')


## 6. Package results for download

Zip the two experiment dirs (best.pt, test_metrics.json, train_history, config) so they can be pulled back and used for ensemble + TTA locally.

In [ ]:
import shutil, glob, os
out = '/kaggle/working/affecta_results'
os.makedirs(out, exist_ok=True)
for e in glob.glob('/kaggle/working/affecta-ai/experiments/EXP-00[78]*'):
    dst = out + '/' + os.path.basename(e)
    shutil.copytree(e, dst)
    print('copied', dst)
!cd /kaggle/working && zip -r -q affecta_results.zip affecta_results
!ls -lh /kaggle/working/affecta_results.zip

**Done.** Download `/kaggle/working/affecta_results.zip` (right panel → Files) and place it in the local project's `experiments/`.